In [ ]:
from __future__ import annotations

import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles
from scipy.stats import ttest_ind

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "08.MF1.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

print(f"Resolved ROI target: {target}")
print(f"Using top-k = {top_k}")
print(f"Raster shape after binning {raster_4d.shape}")

# shape (units, time, images)
raster_3d = np.nanmean(raster_4d, axis=3)
scores = tut.rank_images_by_response(raster_3d)

idx_topk = scores[:top_k]
trial_3d = raster_3d[:, :, idx_topk]
print(f"Trial averaged, all data {raster_3d.shape}")
print(f"Trial averaged, top-k shape {trial_3d.shape}")

In [ ]:
rng = np.random.default_rng(0)
random_idxs = np.stack(
    [rng.choice(scores[top_k:], size=top_k, replace=False) for _ in range(50)],
    axis=0,
)

fig, axes = plt.subplots(1,2, figsize=(10,3))
for rand_idx in random_idxs:
    rr, rrdv = tut.tuning_rdm(raster_3d, rand_idx)
    eigvals, eigvecs = np.linalg.eigh(rr)
    nvecs = eigvecs / np.nanmean(eigvecs, axis=0, keepdims=True)
    
    axes[0].plot(nvecs[:, -1], c='black', alpha=0.1)   # largest component
    # plt.plot(nvecs[:, -2])
    axes[1].plot(eigvals[::-1][:10], c='black', alpha=0.1)

R, rdv = tut.tuning_rdm(raster_3d, idx_topk)
eigvals, eigvecs = np.linalg.eigh(R)
nvecs = eigvecs / np.nanmean(eigvecs, axis=0, keepdims=True)

axes[0].plot(nvecs[:, -1], c='red')   # largest component
# plt.plot(nvecs[:, -2])
axes[1].plot(eigvals[::-1][:10], c='red')
plt.show()